# 체크포인트 아카이브 — Drive ↔ 저장소 양방향

`runs/`에는 **텍스트 산출물(metrics.json · train.log)만** 두고 학습 체크포인트는 Drive 미러에
보관한다. 이 노트북이 그 이동과 **복원**을 담당한다.

**무엇이 어디에 남는가**

| | model.pt | train.log | metrics.json |
|---|---|---|---|
| 저장소 (git) | ✗ — Drive로 이동 (**단 `stage_a`는 유지**) | ✓ **정본** | ✓ **정본** |
| Drive 미러 | ✓ 유일 사본 | ✓ 사본 | ✓ 사본 |

Drive에 3종을 다 두는 것은 학습 중 `train_gpu.py`의 `_mirror_copy`가 이미 그렇게 하기
때문이다 — 미러 디렉토리가 run 하나를 자기완결적으로 담고 있어야 새 VM에서 복원이 된다.
이 노트북은 그 형태를 유지하면서, CPU 경로로 학습해 미러에 없던 run(`mlp_baseline`)까지
채워 **미러를 균일하게** 만든다. 부수 효과로 세션이 죽어 뒤처진 미러 사본도 git 정본으로
갱신된다.

**git 유지 예외 — `runs/stage_a/*/model.pt` (합계 44 KB)**: `scripts/diagnose_calibration.py`가
직접 로드해 `reports/stage_a_gate.md`를 재생성하므로, Drive로 보내면 **문서에 적힌 재현
커맨드가 클론한 사람 누구에게도 안 돌아간다.**

**왜 양방향인가** — push 전용이면 한 번 실행한 뒤 입력이 사라져 다시 못 돌리는 파일이 된다.
재검증이 필요할 때 되돌릴 경로가 있어야 아카이브가 성립한다.

**안전장치** — 업로드 후 `flush_and_unmount()` → 재마운트 → **sha256 재검증**까지 통과해야
성공으로 표시한다. Drive FUSE는 비동기 업로드라 검증 없이 끝내면 구버전이 남을 수 있다
(실사례: 3.4 GB `resume.pt`가 5에폭 뒤처진 채 남았다).

**pull 주의 — 기본값은 `model.pt`만 되돌린다.** 텍스트 2종은 git이 정본이고, 미러 사본은
과거에 뒤처진 이력이 있다 (미러 복원 경로를 탄 run 4개의 `train.log`가 마지막 줄이 빠진
채로 남아 있었다 — `train_gpu.py`의 복사·로그 순서 버그). 되돌리면 정본을 사본으로
덮어쓸 위험이 있으므로 `PULL_TEXT`를 명시적으로 켜야 한다.

**산출물** `runs/CHECKPOINTS.md` — run / 바이트 / sha256 / Drive 경로 / **원본 커밋 SHA**.
git에 추적되므로 "무엇이 어디 있는지"가 저장소에 남고, 삭제 후에도 그 SHA로 히스토리에서
복구할 수 있다 (**Drive는 편의 사본이지 유일본이 아니다** — 단 `winner-repro-asis`는
애초에 커밋된 적이 없어 Drive가 유일본이다).

**사용법**: Colab 런타임에 연결하고 셀 4의 `MODE`를 정한 뒤 위에서부터 실행한다.
`MODE = "push"` 는 체크포인트가 아직 있는 커밋(`REF`)에서 돌려야 한다.

## 1. 환경 · Drive 마운트

In [1]:
import os
import shutil
import subprocess
import sys
from hashlib import sha256
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print("환경:", "Colab VM" if IN_COLAB else "로컬 커널")

if IN_COLAB:
    from google.colab import drive

    # 규약 1 — 항상 force_remount: 이전 세션의 stale 마운트를 배제한다.
    drive.mount("/content/drive", force_remount=True)
    MIRROR_ROOT = Path("/content/drive/MyDrive/FringeNet/runs_mirror")
else:
    # 로컬 테스트용 — rclone 등으로 마운트한 경로를 넣거나 임시 디렉토리로 드라이런한다.
    MIRROR_ROOT = Path(os.environ.get("FRINGENET_MIRROR", "/tmp/fringenet_mirror"))
    drive = None

MIRROR_ROOT.mkdir(parents=True, exist_ok=True)
print("미러 루트:", MIRROR_ROOT)

Mounted at /content/drive
미러 루트: /content/drive/MyDrive/FringeNet/runs_mirror


## 2. 저장소 준비

Colab VM의 파일시스템은 로컬 워크스페이스와 별개다 — `src/`·`runs/`를 VM에 가져와야 한다.
`MODE = "push"` 는 **체크포인트가 아직 있는 커밋**을 봐야 하므로 `REF`로 고정한다.

In [7]:
REPO_URL = "https://github.com/SungHan-Bae/FringeNet.git"
REF = "fix/review-2026-08-13-2"  # push 모드에서는 체크포인트가 남아 있는 브랜치/커밋/태그로 바꿀 것

if IN_COLAB:
    REPO = Path("/content/FringeNet")
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True)
    # fetch + reset --hard: VM clone이 origin과 갈라져도 ff-only pull 거부에 걸리지 않는다.
    subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "--tags", "origin"], check=True)
    target = REF if REF.startswith(("archive/", "v")) or len(REF) == 40 else f"origin/{REF}"
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "--quiet", target], check=True)
else:
    REPO = Path.cwd()
    while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
        REPO = REPO.parent

os.chdir(REPO)
SOURCE_SHA = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("저장소:", REPO, "\n원본 커밋:", SOURCE_SHA[:12])

저장소: /content/FringeNet 
원본 커밋: 2a2ba5673279


## 3. 설정

In [8]:
MODE = "push"          # "push" = 저장소 → Drive | "pull" = Drive → 저장소
TARGET_FILTER = ""     # 예: "flatten-dilated-bound" (빈 문자열이면 전부)
PULL_TEXT = False      # pull에서 train.log·metrics.json도 되돌릴지 — 위 "pull 주의" 참조
PUSH_MANIFEST = True   # 매니페스트를 저장소에 커밋·push (끄면 VM 안에만 남아 사라진다)

# Drive 미러는 run 하나를 자기완결적으로 담는다 (train_gpu.py의 _mirror_copy와 같은 3종).
MIRROR_FILES = ("model.pt", "train.log", "metrics.json")
# 저장소에서 Drive로 **이동**하는 것 = 이것뿐. 나머지 2종은 git이 정본이고 미러는 사본이다.
ARCHIVED_FILE = "model.pt"
GIT_KEPT_EXPERIMENTS = ("stage_a",)  # 헤더의 "git 유지 예외" 참조
MANIFEST = Path("runs/CHECKPOINTS.md")

assert MODE in ("push", "pull"), MODE
print(f"MODE = {MODE}  /  미러 3종 {MIRROR_FILES}  /  git 유지 예외 {GIT_KEPT_EXPERIMENTS}")
print(f"매니페스트 push: {PUSH_MANIFEST}")

MODE = push  /  미러 3종 ('model.pt', 'train.log', 'metrics.json')  /  git 유지 예외 ('stage_a',)


## 4. 공용 함수

In [9]:
CHUNK = 1 << 20


def digest(path: Path) -> str:
    """파일의 sha256 (청크 읽기 — 수백 MB 체크포인트도 메모리를 안 쓴다)."""
    h = sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(CHUNK), b""):
            h.update(block)
    return h.hexdigest()


def atomic_copy(src: Path, dst: Path) -> None:
    """임시파일 → 교체. 복사 도중 세션이 죽어도 목적지가 깨지지 않는다."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)


def targets() -> list[tuple[str, str]]:
    """아카이브 대상 (실험, run). metrics.json이 있는 run 디렉토리를 기준으로 하므로
    체크포인트를 이미 삭제한 뒤에도 같은 목록이 나온다 (pull이 성립하는 이유)."""
    out = []
    for metrics in sorted(Path("runs").glob("*/*/metrics.json")):
        experiment, run = metrics.parent.parent.name, metrics.parent.name
        if experiment in GIT_KEPT_EXPERIMENTS:
            continue
        if TARGET_FILTER and TARGET_FILTER not in run:
            continue
        out.append((experiment, run))
    return out


def paths_for(experiment: str, run: str, name: str) -> tuple[Path, Path]:
    """(저장소 경로, 미러 경로)."""
    return (Path("runs") / experiment / run / name,
            MIRROR_ROOT / experiment / run / name)


def remount() -> None:
    """대기 중인 업로드를 끝내고 다시 마운트한다 — 검증이 캐시가 아니라 Drive를 보게 한다."""
    if drive is None:
        return
    drive.flush_and_unmount()
    drive.mount("/content/drive", force_remount=True)


def human(n: int) -> str:
    return f"{n / 1048576:.1f} MB" if n >= 1048576 else f"{n / 1024:.0f} KB"

## 5. 계획 확인 (실행 전 무엇이 움직이는지 본다)

In [10]:
print(f"{'실험/run':45s} {'저장소':>12s} {'Drive':>12s} {'model.pt':>10s}")
total = 0
for experiment, run in targets():
    have_local = [n for n in MIRROR_FILES if paths_for(experiment, run, n)[0].exists()]
    have_mirror = [n for n in MIRROR_FILES if paths_for(experiment, run, n)[1].exists()]
    local_ckpt, mirror_ckpt = paths_for(experiment, run, ARCHIVED_FILE)
    size = (local_ckpt.stat().st_size if local_ckpt.exists()
            else mirror_ckpt.stat().st_size if mirror_ckpt.exists() else 0)
    total += size
    print(f"{experiment + '/' + run:45s} {f'{len(have_local)}/3':>12s} "
          f"{f'{len(have_mirror)}/3':>12s} {human(size):>10s}")

assert targets(), (
    "아카이브 대상이 0개다 — REF가 runs/를 담은 커밋인지, TARGET_FILTER가 과하지 않은지 확인. "
    "이대로 진행하면 매니페스트가 만들어지지 않는다."
)
print(f"\nmodel.pt 합계 {human(total)}")
print(f"git 유지 예외: stage_a {sum(1 for _ in Path('runs').glob('stage_a/*/model.pt'))}개 "
      f"(diagnose 스크립트가 직접 읽는다)")

실험/run                                                 저장소        Drive   model.pt
level1_cnn/dilated                                     3/3          3/3     2.5 MB
level1_cnn/flatten                                     3/3          3/3     2.5 MB
level1_cnn/flatten-dilated                             3/3          3/3     2.5 MB
level1_cnn/flatten-dilated-bound                       3/3          3/3     2.5 MB
level1_cnn/single-scale                                3/3          1/3     2.5 MB
level1_cnn/single-scale-shuffled                       3/3          1/3     2.5 MB
mlp_baseline/dropout0.0                                3/3          1/3     2.5 MB
mlp_baseline/dropout0.1                                3/3          1/3     2.5 MB
strong_baseline/winner-repro-asis                      2/3          3/3   813.6 MB

model.pt 합계 833.7 MB
git 유지 예외: stage_a 10개 (diagnose 스크립트가 직접 읽는다)


## 6. PUSH — 저장소 → Drive (3종), 그리고 **재검증**

`업로드 → flush/재마운트 → Drive 사본 재해시 → 로컬 해시와 대조`를 파일마다 거친다.
하나라도 불일치하면 assert로 멈춘다 — 삭제 커밋으로 넘어가서는 안 되기 때문이다.

In [11]:
records: list[dict] = []

if MODE == "push":
    staged = []  # (실험, run, 파일명, 기대 sha, 바이트, 상태)
    for experiment, run in targets():
        for name in MIRROR_FILES:
            local, mirror = paths_for(experiment, run, name)
            if local.exists():
                atomic_copy(local, mirror)
                staged.append((experiment, run, name, digest(local),
                               local.stat().st_size, "uploaded"))
            elif mirror.exists():
                # 저장소에 없고 미러에만 있는 것 = 애초에 커밋된 적 없는 Drive 유일본
                # (예: 813 MB winner 체크포인트). 해시만 등록한다.
                staged.append((experiment, run, name, digest(mirror),
                               mirror.stat().st_size, "drive-only"))
            else:
                staged.append((experiment, run, name, "", 0, "missing"))
        flags = " ".join(s[5][:2] for s in staged[-len(MIRROR_FILES):])
        print(f"[up ] {experiment}/{run:34s} {flags}")

    print("\n대기 업로드 flush 후 재마운트…")
    remount()

    for experiment, run, name, expect_sha, size, status in staged:
        _, mirror = paths_for(experiment, run, name)
        if status == "missing":
            verdict = "MISSING"
        elif not mirror.exists():
            verdict = "FAILED (미러에 파일 없음)"
        else:
            got = digest(mirror)
            verdict = "OK" if got == expect_sha else f"FAILED (sha {got[:12]} ≠ {expect_sha[:12]})"
        records.append({"experiment": experiment, "run": run, "file": name, "bytes": size,
                        "sha256": expect_sha, "status": status, "verify": verdict})

    for experiment, run in targets():
        rows = [r for r in records if (r["experiment"], r["run"]) == (experiment, run)]
        ok = sum(r["verify"] == "OK" for r in rows)
        gone = [r["file"] for r in rows if r["status"] == "missing"]
        note = f"  (없음: {', '.join(gone)})" if gone else ""
        print(f"[ver] {experiment}/{run:34s} {ok}/{len(rows)} OK{note}")

    ok_all = sum(r["verify"] == "OK" for r in records)
    expected = sum(r["status"] != "missing" for r in records)
    print(f"\n검증 통과 {ok_all}/{expected} (대상 파일 {len(records)}개)")
    assert ok_all == expected, "재검증 실패가 있다 — 삭제 커밋으로 넘어가지 말 것"

Mounted at /content/drive
[ver] level1_cnn/dilated                            3/3 OK
[ver] level1_cnn/flatten                            3/3 OK
[ver] level1_cnn/flatten-dilated                    3/3 OK
[ver] level1_cnn/flatten-dilated-bound              3/3 OK
[ver] level1_cnn/single-scale                       3/3 OK
[ver] level1_cnn/single-scale-shuffled              3/3 OK
[ver] mlp_baseline/dropout0.0                         3/3 OK
[ver] mlp_baseline/dropout0.1                         3/3 OK
[ver] strong_baseline/winner-repro-asis                  3/3 OK

검증 통과 27/27 (대상 파일 27개)


## 7. 매니페스트 — `runs/CHECKPOINTS.md`

행은 run 단위다. **sha256은 `model.pt`의 것**(Drive에만 있는 유일 사본)이고, 텍스트 2종은
git이 정본이라 검증 결과만 싣는다. 삭제 후 복구는 이 파일의 **원본 커밋 SHA**로
`git show <SHA>:runs/.../model.pt` 하면 되므로 Drive 유일본 항목만 진짜 외부 의존이다.

In [12]:
if MODE == "push" and records:
    lines = [
        "# 체크포인트 아카이브 (`notebooks/checkpoint_archive.ipynb` 산출 — 손으로 고치지 말 것)",
        "",
        "`runs/`에는 텍스트 산출물(`train.log` · `metrics.json`)만 두고 학습 체크포인트는 "
        "Drive 미러에 보관한다. **미러에는 3종을 다 둔다** — run 하나가 자기완결적이어야 새 VM "
        "에서 복원이 되고, 학습 중 `train_gpu.py`가 이미 그 형태로 쓰기 때문이다. "
        "텍스트 2종은 **git이 정본**이고 미러는 사본이다.",
        "",
        f"미러 루트: `{MIRROR_ROOT.as_posix()}/<실험>/<run>/`",
        "",
        "**예외 — `runs/stage_a/*/model.pt`는 git에 남긴다** (합계 44 KB): "
        "`scripts/diagnose_calibration.py`가 직접 로드해 `reports/stage_a_gate.md`를 "
        "재생성하므로 Drive로 보내면 문서의 재현 커맨드가 깨진다.",
        "",
        f"- 원본 커밋: `{SOURCE_SHA}`",
        "- 복구: `git show <원본 커밋>:runs/<실험>/<run>/model.pt > model.pt` — 히스토리에 "
        "blob이 남아 있으므로 **Drive는 편의 사본이다**. 단 `drive-only` 항목은 커밋된 적이 "
        "없어 Drive가 유일본이다.",
        "- 되돌리기: 이 노트북을 `MODE = \"pull\"` 로 실행 (기본은 `model.pt`만 — 텍스트 2종은 "
        "git 정본을 덮어쓰지 않도록 `PULL_TEXT`로 명시해야 한다)",
        "",
        "| 실험 | run | model.pt | sha256 (model.pt) | 상태 | 미러 3종 검증 |",
        "|---|---|---|---|---|---|",
    ]
    for experiment, run in targets():
        rows = [r for r in records if (r["experiment"], r["run"]) == (experiment, run)]
        ckpt = next(r for r in rows if r["file"] == ARCHIVED_FILE)
        ok = sum(r["verify"] == "OK" for r in rows)
        gone = [r["file"] for r in rows if r["status"] == "missing"]
        cell = f"{ok}/{len(rows)} OK" + (f" ({', '.join(gone)} 없음)" if gone else "")
        sha = f"`{ckpt['sha256'][:16]}…`" if ckpt["sha256"] else "—"
        lines.append(
            f"| {experiment} | `{run}` | {human(ckpt['bytes']) if ckpt['bytes'] else '—'} "
            f"| {sha} | {ckpt['status']} | {cell} |"
        )
    ckpt_total = sum(r["bytes"] for r in records if r["file"] == ARCHIVED_FILE)
    lines += ["", f"model.pt 합계 {human(ckpt_total)} / run {len(targets())}개 "
                  f"(미러 파일 {sum(r['status'] != 'missing' for r in records)}개)", ""]
    body = "\n".join(lines)
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    MANIFEST.write_text(body, encoding="utf-8")
    # Drive에도 같은 사본을 둔다 — 런타임이 죽어도 남고, 미러 내용을 미러 옆에서 볼 수 있다.
    drive_copy = MIRROR_ROOT / "CHECKPOINTS.md"
    drive_copy.write_text(body, encoding="utf-8")
    print(body)
    print(f"\n기록 위치\n  VM   : {MANIFEST.resolve()}\n  Drive: {drive_copy}")
    print("  → VM 사본은 런타임과 함께 사라진다. 아래 push 셀이 저장소에 반영한다."
          if PUSH_MANIFEST else
          "  → PUSH_MANIFEST = False 이므로 저장소에 반영되지 않는다. Drive 사본을 쓸 것.")
elif MODE == "push":
    print("[!!] records가 비어 있어 매니페스트를 만들지 않았다 — 위 push 셀이 실행됐는지 확인.")
else:
    print(f"매니페스트는 push 모드에서만 만든다 (현재 MODE = {MODE}).")

# 체크포인트 아카이브 (`notebooks/checkpoint_archive.ipynb` 산출 — 손으로 고치지 말 것)

`runs/`에는 텍스트 산출물(`train.log` · `metrics.json`)만 두고 학습 체크포인트는 Drive 미러에 보관한다. **미러에는 3종을 다 둔다** — run 하나가 자기완결적이어야 새 VM 에서 복원이 되고, 학습 중 `train_gpu.py`가 이미 그 형태로 쓰기 때문이다. 텍스트 2종은 **git이 정본**이고 미러는 사본이다.

미러 루트: `/content/drive/MyDrive/FringeNet/runs_mirror/<실험>/<run>/`

**예외 — `runs/stage_a/*/model.pt`는 git에 남긴다** (합계 44 KB): `scripts/diagnose_calibration.py`가 직접 로드해 `reports/stage_a_gate.md`를 재생성하므로 Drive로 보내면 문서의 재현 커맨드가 깨진다.

- 원본 커밋: `2a2ba5673279dae9aa9600036b07c432cde8440d`
- 복구: `git show <원본 커밋>:runs/<실험>/<run>/model.pt > model.pt` — 히스토리에 blob이 남아 있으므로 **Drive는 편의 사본이다**. 단 `drive-only` 항목은 커밋된 적이 없어 Drive가 유일본이다.
- 되돌리기: 이 노트북을 `MODE = "pull"` 로 실행 (기본은 `model.pt`만 — 텍스트 2종은 git 정본을 덮어쓰지 않도록 `PULL_TEXT`로 명시해야 한다)

| 실험 | run | model.pt | sha256 (model.pt) | 상태 | 미러 3종 검증 |
|---|---|---|---|---|---|
| level1_cnn | `dilated` | 2.5 MB | `958d8847f7821867…` | uploaded | 3/3 OK |
| level1_cnn | `flatten` | 2.5 MB

## 8. PULL — Drive → 저장소 (매니페스트의 sha256으로 검증)

체크포인트를 지운 뒤 재검증이 필요할 때 쓴다. 해시가 어긋나면 받은 파일을 남기지 않는다.
텍스트 2종은 `PULL_TEXT = True`로만 되돌린다 — **git 정본을 미러 사본으로 덮어쓸 위험**이
있기 때문이다 (과거 미러의 `train.log`가 마지막 줄이 빠진 채 남아 있던 이력).

In [13]:
if MODE == "pull":
    import re

    want = {}
    if MANIFEST.exists():
        for m in re.finditer(r"^\| (\S+) \| `(\S+)` \|[^|]*\| `([0-9a-f]{16})",
                             MANIFEST.read_text(encoding="utf-8"), re.M):
            want[(m.group(1), m.group(2))] = m.group(3)
        print(f"매니페스트에서 model.pt 해시 {len(want)}개 읽음")
    else:
        print(f"[warn] {MANIFEST} 없음 — 해시 대조 없이 복사만 한다")

    names = MIRROR_FILES if PULL_TEXT else (ARCHIVED_FILE,)
    print(f"복원 대상 파일: {names}\n")
    for experiment, run in targets():
        for name in names:
            local, mirror = paths_for(experiment, run, name)
            if not mirror.exists():
                print(f"[skip] {experiment}/{run}/{name}  미러에 없음")
                continue
            atomic_copy(mirror, local)
            expect = want.get((experiment, run)) if name == ARCHIVED_FILE else None
            got = digest(local)
            if expect and not got.startswith(expect):
                local.unlink()
                print(f"[!! ] {experiment}/{run}/{name}  sha 불일치 "
                      f"({got[:16]} ≠ {expect}) — 삭제했다")
            else:
                print(f"[dl ] {experiment}/{run}/{name}  {human(local.stat().st_size)}  "
                      f"{'해시 일치' if expect else '해시 미확인'}")

## 9. 매니페스트 커밋·push (선택)

PAT는 규약 2대로 정적 소스에서 자동 로드한다 (env → Colab Secrets → Drive 파일).
체크포인트 삭제는 이 노트북이 하지 않는다 — 검증이 통과한 뒤 별도 커밋으로 한다.

In [15]:
def load_pat() -> str | None:
    if os.environ.get("GITHUB_PAT"):
        return os.environ["GITHUB_PAT"]
    try:
        from google.colab import userdata

        if token := userdata.get("GITHUB_PAT"):
            return token
    except Exception:
        pass
    secret = Path("/content/drive/MyDrive/FringeNet/secrets/github_pat.txt")
    return secret.read_text().strip() if secret.exists() else None


if PUSH_MANIFEST and MODE == "push":
    pat = load_pat()
    assert pat, "PAT를 찾지 못했다 (env GITHUB_PAT / Colab Secrets / Drive secrets 파일)"
    subprocess.run(["git", "config", "user.email", "bshzz1006@hits.ai"], check=True)
    subprocess.run(["git", "config", "user.name", "Bae-SungHan"], check=True)
    subprocess.run(["git", "add", str(MANIFEST)], check=True)
    subprocess.run(["git", "commit", "-m",
                    "docs: 체크포인트 아카이브 매니페스트 (Drive 미러 3종 + sha256 검증)"],
                   check=True)
    subprocess.run(["git", "push", REPO_URL.replace("https://", f"https://{pat}@"),
                    f"HEAD:{REF}"], check=True)
    print("push 완료")
elif MODE == "push":
    print("매니페스트 push 생략 (PUSH_MANIFEST = False) — "
          f"Drive 사본만 남는다: {MIRROR_ROOT / 'CHECKPOINTS.md'}")

매니페스트 push 생략 (PUSH_MANIFEST = False)


## 다음 단계

1. 위 **검증 통과 N/N** 을 확인한다 — 하나라도 FAILED면 삭제 커밋으로 넘어가지 않는다.
2. `runs/CHECKPOINTS.md` 가 저장소에 올라갔는지 확인한다 (`PUSH_MANIFEST = True`면 자동).
   **이 노트북은 Colab VM 안에서 돈다** — VM의 `/content/FringeNet/runs/CHECKPOINTS.md`는
   런타임과 함께 사라지므로, push하지 않았다면 Drive 루트의 `CHECKPOINTS.md` 사본을 쓴다.
3. 그 다음에 `model.pt`를 git에서 제거하고 `.gitignore`·규약을 갱신한다 (별도 커밋).
   **그 커밋에서 이 노트북도 삭제한다** — 일회성 이관 도구이고, 이후 학습에서 나오는
   체크포인트는 `train_gpu.py`의 `_mirror_copy`가 학습 중에 이미 Drive로 보내므로
   다시 쓸 일이 없다. 되돌리려면 `git show <원본 커밋>:runs/<실험>/<run>/model.pt` 로 충분하다.

런타임 반납은 두지 않았다 — 대화형 유틸리티이고 pull로 되돌린 뒤 바로 이어 쓰는 경우가 많다.
끝났으면 런타임 메뉴에서 직접 종료할 것.